# Protein Identification

From the diffential abundance analysis, some taxa were particularly enriched or depleated from the lifestyle groups. We decided to pick out 1 particulary enriched and 1 depleated taxa when comparing the the groups "Western" and "Fisher".

Here are the taxonomy with the respective feature ID:

bee4ccf1-9015-4cfb-82a3-105a91e5acb9 : d__Bacteria;k__Bacillati;p__Actinomycetota;c__Actinomycetes;o__Bifidobacteriales;f__Bifidobacteriaceae;g__Bifidobacterium;s__Bifidobacterium_bifidum

10902617-4ae8-45aa-b3cd-fae31158d31b: d__Bacteria;k__Pseudomonadati;p__Pseudomonadota;c__Gammaproteobacteria;o__Aeromonadales;f__Succinivibrionaceae;g__Succinivibrio;s__Succinivibrio_dextrinosolvens


First let's setup the notebook.

In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd
import qiime2 as q2
from qiime2 import Visualization

In [ ]:
data_dir = 'updog_data'
os.makedirs(data_dir, exist_ok=True)

Creating the dataframe with the relevant feature IDs and save it as a tsv table:

In [ ]:
df_ids = pd.DataFrame({'FeatureID': ['bee4ccf1-9015-4cfb-82a3-105a91e5acb9',
                                     '10902617-4ae8-45aa-b3cd-fae31158d31b']})


In [ ]:
df_ids.to_csv(f'{data_dir}/ids.tsv', sep="\t", index=False)

## 1. Filter out the selected MAGs from the dereplicated MAGs file

We then select only the two feature IDs out of the dereplicated MAGs file, using the table we created before.

**the following command was run on Euler**

In [ ]:
! qiime feature-table filter-features \
    --i-table $data_dir/mags_derep_all_domains.qza \
    --m-metadata-file $data_dir/ids.tsv \
    --p-no-exclude-ids \
    --o-filtered-table $data_dir/mags-filtered-for-proteins.qza

Now we can import the generated file from the previous command for the next steps on the notebook:

In [ ]:
!wget -O "$data_dir/mags-filtered-for-proteins.qza" "https://polybox.ethz.ch/index.php/s/sCy8P9MbW9TWXQr"

In [ ]:
! qiime tools peek $data_dir/mags-filtered-for-proteins.qza

## 2. Prediction of coded proteins

The `annotate predict-genes-prodigal` command scans a prokaryotic genome (or in this case, a MAG) and identifies all open reading frames that look like protein-coding genes.

In [ ]:
! qiime annotate predict-genes-prodigal \
    --i-seqs $data_dir/mags-filtered-for-proteins.qza \
    --o-loci $data_dir/updog-loci.qza \
    --o-genes $data_dir/updog-genes.qza \
    --o-proteins $data_dir/updog-proteins.qza 

We then export the updog-proteins.qza file to obtain the fasta files of the protein preediction from both MAGs.

In [ ]:
! qiime tools export \
  --input-path $data_dir/updog-proteins.qza  \
  --output-path $data_dir/updog-proteins

3. Import fasta files from relevant proteins

From [_this paper_](https://doi.org/10.1016/j.cub.2015.04.055) from Rampelli et al., we decided to focus on the proteins that were the most significantly different between the two populations they focused on. We then imported the fasta files of all the relevant proteins from the [NCBi protein database](https://www.ncbi.nlm.nih.gov/protein/). Once imported, we merge them into one file.

In [ ]:
# Combine sequences into one file
!cat \
  $data_dir/fasta-enzymes/dehydratase.fasta \
  $data_dir/fasta-enzymes/formiminotransferase.fasta \
  $data_dir/fasta-enzymes/fructofuranosidase.fasta \
  $data_dir/fasta-enzymes/glucosidase.fasta \
  $data_dir/fasta-enzymes/xylosidase.fasta \
  > $data_dir/fasta-enzymes/all_enzymes.fasta

We then use the merged file and set it as our database.

In [ ]:
!makeblastdb \
   -in $data_dir/fasta-enzymes/all_enzymes.fasta \
   -dbtype prot \
   -out $data_dir/updog_protein_db/updog_protein_db/

## 3. Blasting 

The fasta files from both relevant MAGs are then blasted with the database we previously created. The outputs are .txt files that are then compared with each other.

First for the enriched MAG in the "Western" group (depleted in the "Fisher" group).

In [ ]:
!blastp \
   -query $data_dir/updog-proteins/10902617-4ae8-45aa-b3cd-fae31158d31b.fasta \
   -db $data_dir/updog_protein_db \
   -out $data_dir/results-10902617-4ae8-45aa-b3cd-fae31158d31b.txt \
   -evalue 1e-5 \
   -outfmt 6 

then for the enriched MAG in the "Fisher" group (depleted in the "Wester" group).

In [ ]:
!blastp \
   -query $data_dir/updog-proteins/bee4ccf1-9015-4cfb-82a3-105a91e5acb9.fasta \
   -db $data_dir/updog_protein_db \
   -out $data_dir/results-bee4ccf1-9015-4cfb-82a3-105a91e5acb9.txt \
   -evalue 1e-5 \
   -outfmt 6 

## 4. Comparing our results

In [ ]:
import pandas as pd

cols = [
    "qseqid","sseqid","pident","length","mismatch","gapopen",
    "qstart","qend","sstart","send","evalue","bitscore"
]

# load both BLAST tables
df1 = pd.read_csv(f"{data_dir}/results-10902617-4ae8-45aa-b3cd-fae31158d31b.txt", sep="\t", names=cols)
df2 = pd.read_csv(f"{data_dir}/results-bee4ccf1-9015-4cfb-82a3-105a91e5acb9.txt", sep="\t", names=cols)

# keep only the best hit per query for each MAG
df1_best = df1.sort_values("evalue").groupby("sseqid").first().reset_index()
df2_best = df2.sort_values("evalue").groupby("sseqid").first().reset_index()


# add MAG labels
df1_labeled = df1_best.copy()
df1_labeled["MAG"] = "MAG1"

df2_labeled = df2_best.copy()
df2_labeled["MAG"] = "MAG2"

# combine them into one table
combined = pd.concat([df1_labeled, df2_labeled], ignore_index=True)

combined

In [ ]:
combined.to_csv(f'{data_dir}/combined.tsv', sep="\t", index=False)